# Rdakt AI + PydanticAI Integration

Three ways to add PII anonymization to your PydanticAI agents — pick the tier that fits your use case.

| Tier | API | Interception layer | Best for |
|------|-----|--------------------|----------|
| 1 | `create_http_client()` | HTTP transport | Full control over provider setup |
| 2 | `protect_agent()` | HTTP transport | Quick wrap of an existing agent |
| 3 | `RdaktCapability` | Message abstraction | Native PydanticAI integration |

## Setup

Load a config from `examples/configs/`. Change the path to try different behaviors:

| Config | What it does |
|--------|-------------|
| `minimal.yaml` | Regex detection, in-memory store (default) |
| `audit.yaml` | Detects PII and logs it but forwards unanonymized text |
| `production.yaml` | Regex + NER, SQLite persistence, fail-closed |
| `ner_only.yaml` | spaCy NER without regex (requires `rdakt-ai[ner]`) |
| `redis.yaml` | Redis-backed session store for distributed deployments |

In [ ]:
import os
from pathlib import Path

# Set your API key (or export OPENAI_API_KEY in your shell)
os.environ.setdefault("OPENAI_API_KEY", "sk-your-key-here")

from rdakt_ai.config import load_config

# ── Change this path to switch configs ──
CONFIG_FILE = Path("configs/minimal.yaml")

config = load_config(CONFIG_FILE)
print(f"Loaded config: {CONFIG_FILE}")
print(f"  mode: {config.mode}")
print(f"  pipeline: {config.pipeline}")
print(f"  session store: {config.session_store}")

---
## Tier 1: Low-level — `create_http_client()`

Returns a wired `httpx.AsyncClient` with Rdakt middleware as the transport.
You construct the provider and model yourself — maximum control.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from rdakt_ai.integrations.pydantic_ai import create_http_client

http_client = create_http_client(config=config, session_key="tier1-conv")

agent_tier1 = Agent(
    OpenAIChatModel("gpt-4o-mini", provider=OpenAIProvider(http_client=http_client)),
    system_prompt="You are a helpful assistant.",
)

print("Agent created with Rdakt transport")
print(f"Transport type: {type(http_client._transport).__name__}")

In [ ]:
result = await agent_tier1.run("My email is john.doe@acme.com and my SSN is 123-45-6789")
print(result.output)

---
## Tier 2: High-level — `protect_agent()`

Wraps an existing agent in one call. Reconstructs the provider with a Rdakt-wired
`http_client` under the hood — no manual provider setup needed.

In [ ]:
from pydantic_ai import Agent

from rdakt_ai.integrations.pydantic_ai import protect_agent

# Create a normal agent
agent = Agent("openai:gpt-4o-mini", system_prompt="You are a helpful assistant.")

# Wrap it — returns a new agent, original is untouched
agent_tier2 = protect_agent(agent, config=config, session_key="tier2-conv")

print(f"Original agent transport: {type(agent.model._provider._client._client._transport).__name__}")
print(f"Protected agent transport: {type(agent_tier2.model._provider._client._client._transport).__name__}")

In [ ]:
result = await agent_tier2.run("My phone number is (555) 123-4567 and I live at 742 Evergreen Terrace")
print(result.output)

---
## Tier 3: Message-level — `RdaktCapability`

Native PydanticAI capability. Anonymizes message text *before* it reaches the model
and deanonymizes the response *after* — no httpx transport involved.

This is the cleanest integration: just add it to `capabilities=[]`.

In [ ]:
from pydantic_ai import Agent

from rdakt_ai.integrations.pydantic_ai import RdaktCapability

agent_tier3 = Agent(
    "openai:gpt-4o-mini",
    system_prompt="You are a helpful assistant.",
    capabilities=[RdaktCapability(config=config, session_key="tier3-conv")],
)

print("Agent created with RdaktCapability")

In [ ]:
result = await agent_tier3.run("My credit card is 4111-1111-1111-1111 and my name is Jane Smith")
print(result.output)

---
## Multi-turn with session persistence

Pass a `session_key` to reuse the same entity map across turns.
The same PII gets the same token every time.

In [ ]:
from rdakt_ai.stores import MemoryStore

store = MemoryStore()

agent_multi = Agent(
    "openai:gpt-4o-mini",
    system_prompt="You are a helpful assistant. Remember details from our conversation.",
    capabilities=[RdaktCapability(config=config, session_key="multi-turn", store=store)],
)

# Turn 1
r1 = await agent_multi.run("My name is Alice and my email is alice@example.com")
print("Turn 1:", r1.output)

# Turn 2 — same session, tokens are reused
r2 = await agent_multi.run("What was my email again?")
print("Turn 2:", r2.output)